# scVI default - cellranger_versions

Seeds: `0-9`

Metrics: BAMI, CASW, SCGRAPHM, BRAS, KBET, CPCR

This notebook is intentionally unexecuted.


In [1]:
import gc
import inspect
import json
import os
import sys
from pathlib import Path

os.environ["PYTHONNOUSERSITE"] = "1"
PROJECT_ROOT = (
    "/ubc/cs/research/beaver/projects/"
    "Single-cell-integration-metrics"
)
os.environ["R_HOME"] = (
    f"{PROJECT_ROOT}/CondaEnv/scvi_py310/lib/R"
)
os.environ["R_LIBS_SITE"] = (
    f"{PROJECT_ROOT}/CondaEnv/harmony_optuna/lib/R/library"
)
sys.path.insert(0, f"{PROJECT_ROOT}/scib")

OPTIMIZER_PATH = Path(
    PROJECT_ROOT,
    "scripts",
    "optuna_multi_metrics",
    "optuna_scvi",
    "optimize_scvi_multi_metrics.py",
)
sys.path.insert(0, str(OPTIMIZER_PATH.parent))

import anndata as ad
import numpy as np
import pandas as pd
import torch
import optimize_scvi_multi_metrics as scvi_workflow

assert Path(scvi_workflow.__file__).resolve() == OPTIMIZER_PATH


In [2]:
DATASET_NAME = 'cellranger_versions'
DATA_PATH = f"{PROJECT_ROOT}/data/{DATASET_NAME}.h5ad"
BATCH_KEYS = ('batch',)
BATCH_KEY = BATCH_KEYS[0]
LABEL_KEY = 'cell_type'

SELECTED_BIO_METRICS = ('BAMI', 'CASW', 'SCGRAPHM')
SELECTED_BATCH_METRICS = ('BRAS', 'KBET', 'CPCR')
SELECTED_METRICS = SELECTED_BIO_METRICS + SELECTED_BATCH_METRICS

N_SEEDS = 10
N_NEIGHBORS = 15
USE_HVG = True
N_TOP_HVG = 2000
MAX_EPOCHS = 400

EMBED_KEY = "X_scvi"
CLUSTER_KEY = "leiden_opt"
METRIC_BATCH_KEY = "scvi_metric_batch"
METRIC_LABEL_KEY = "scvi_metric_label"

assert DATASET_NAME == 'cellranger_versions'
assert BATCH_KEYS == ('batch',)
assert LABEL_KEY == 'cell_type'
assert SELECTED_METRICS == ('BAMI', 'CASW', 'SCGRAPHM', 'BRAS', 'KBET', 'CPCR')

pd.Series(
    {
        "dataset": DATASET_NAME,
        "data_path": DATA_PATH,
        "batch_keys": list(BATCH_KEYS),
        "label_key": LABEL_KEY,
        "metrics": list(SELECTED_METRICS),
        "seeds": list(range(N_SEEDS)),
        "use_hvg": USE_HVG,
        "n_top_hvg": N_TOP_HVG,
        "max_epochs": MAX_EPOCHS,
    },
    name="configuration",
)


dataset                                     cellranger_versions
data_path     /ubc/cs/research/beaver/projects/Single-cell-i...
batch_keys                                              [batch]
label_key                                             cell_type
metrics                [BAMI, CASW, SCGRAPHM, BRAS, KBET, CPCR]
seeds                            [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
use_hvg                                                    True
n_top_hvg                                                  2000
max_epochs                                                  400
Name: configuration, dtype: object

In [3]:
scvi_workflow.DATA_PATH = DATA_PATH
scvi_workflow.DATASET_NAME = DATASET_NAME
scvi_workflow.BATCH_KEYS = list(BATCH_KEYS)
scvi_workflow.BATCH_KEY = BATCH_KEY
scvi_workflow.LABEL_KEY = LABEL_KEY
scvi_workflow.USE_HVG = USE_HVG
scvi_workflow.N_TOP_HVG = N_TOP_HVG
scvi_workflow.MAX_EPOCHS = MAX_EPOCHS
scvi_workflow.N_NEIGHBORS = N_NEIGHBORS
scvi_workflow.EMBED_KEY = EMBED_KEY
scvi_workflow.CLUSTER_KEY = CLUSTER_KEY
scvi_workflow.METRIC_BATCH_KEY = METRIC_BATCH_KEY
scvi_workflow.METRIC_LABEL_KEY = METRIC_LABEL_KEY

scvi_workflow.USE_BAMI = "BAMI" in SELECTED_METRICS
scvi_workflow.USE_CASW = "CASW" in SELECTED_METRICS
scvi_workflow.USE_SCGRAPHM = "SCGRAPHM" in SELECTED_METRICS
scvi_workflow.USE_TRAJ = "TRAJ" in SELECTED_METRICS
scvi_workflow.USE_ISOLISI = "ISOLISI" in SELECTED_METRICS
scvi_workflow.USE_BRAS = "BRAS" in SELECTED_METRICS
scvi_workflow.USE_KBET = "KBET" in SELECTED_METRICS
scvi_workflow.USE_CPCR = "CPCR" in SELECTED_METRICS

scvi_workflow.validate_configuration(SELECTED_METRICS)
scvi_workflow.validate_dataset_contents()

pd.Series(
    {
        "optimizer": str(OPTIMIZER_PATH),
        "scvi_version": scvi_workflow.scvi.__version__,
        "cuda_available": torch.cuda.is_available(),
    },
    name="preflight",
)


Dataset preflight: n_obs=19202 n_batches=2 n_labels=15 counts_layer=True
CPCR preflight: eligible_labels=12


optimizer         /ubc/cs/research/beaver/projects/Single-cell-i...
scvi_version                                                  1.3.3
cuda_available                                                 True
Name: preflight, dtype: object

In [4]:
constructor_signature = inspect.signature(
    scvi_workflow.scvi.model.SCVI.__init__
)
training_signature = inspect.signature(
    scvi_workflow.scvi.train.TrainingPlan.__init__
)
assert scvi_workflow.scvi.__version__ == "1.3.3"
assert constructor_signature.parameters["n_latent"].default == 10
assert constructor_signature.parameters["n_hidden"].default == 128
assert constructor_signature.parameters["n_layers"].default == 1
assert constructor_signature.parameters["dropout_rate"].default == 0.1
assert training_signature.parameters["lr"].default == 0.001

SCVI_PARAMS = {'n_latent': 10, 'n_hidden': 128, 'n_layers': 1, 'dropout_rate': 0.1, 'lr': 0.001}
assert tuple(SCVI_PARAMS) == ('n_latent', 'n_hidden', 'n_layers', 'dropout_rate', 'lr')

pd.Series(
    {
        **SCVI_PARAMS,
        "gene_likelihood": "zinb",
        "dispersion": "gene",
        "latent_distribution": "normal",
        "early_stopping": True,
        "max_epochs": MAX_EPOCHS,
        "use_hvg": USE_HVG,
        "n_top_hvg": N_TOP_HVG,
    },
    name="default_setting",
)

def run_scvi_setting(adata, seed):
    return scvi_workflow.run_scvi(
        adata,
        n_latent=SCVI_PARAMS["n_latent"],
        n_hidden=SCVI_PARAMS["n_hidden"],
        n_layers=SCVI_PARAMS["n_layers"],
        dropout_rate=SCVI_PARAMS["dropout_rate"],
        lr=SCVI_PARAMS["lr"],
        seed=int(seed),
    )


In [5]:
def compute_joint_metrics(adata_raw, adata_int):
    result = scvi_workflow.compute_joint_metrics(adata_raw, adata_int)
    if tuple(result) != (
        *SELECTED_METRICS,
        "bio_score",
        "batch_score",
        "joint_score",
    ):
        raise ValueError(f"Unexpected metric result keys: {tuple(result)}")
    return result


In [6]:
results = []

for seed in range(N_SEEDS):
    print(f"\n===== seed {seed} =====", flush=True)

    adata_raw = ad.read_h5ad(DATA_PATH)
    adata_int = run_scvi_setting(adata_raw, seed=seed)
    metric_result = compute_joint_metrics(adata_raw, adata_int)

    row = {"seed": seed, **metric_result}
    results.append(row)
    print(pd.Series(row), flush=True)

    del adata_raw, adata_int
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

results_df = pd.DataFrame(results)
results_df



===== seed 0 =====


[rank: 0] Seed set to 0


/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/scvi_py310/lib/python3.10/site-packages/scvi/dataloaders/_data_splitting.py:258: UserWarning: 2 cells moved from training set to validation set. if you want to avoid it please use train_size parameter during train.
  self.n_train, self.n_val = validate_data_split(
GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


...

Cluster for leiden_opt_2.0 with leiden


resolution: 2.0, bAMI: 0.807358372677404


optimised clustering against scvi_metric_label
optimal cluster resolution: 0.4
optimal score: 0.8356177852630436
BRAS...


seed           9.000000
BAMI           0.835618
CASW           0.590490
SCGRAPHM       0.783047
BRAS           0.900455
KBET           0.700615
CPCR           0.971411
bio_score      0.736385
batch_score    0.857494
joint_score    0.784828
dtype: float64


,seed,BAMI,CASW,SCGRAPHM,BRAS,KBET,CPCR,bio_score,batch_score,joint_score
0,0,0.843123,0.580375,0.766145,0.908906,0.688223,0.970033,0.729881,0.855721,0.780217
1,1,0.831058,0.583417,0.755485,0.900468,0.680769,0.972624,0.723320,0.851287,0.774507
2,2,0.847728,0.588101,0.780758,0.901932,0.707044,0.972917,0.738862,0.860631,0.787570
3,3,0.840109,0.589590,0.776081,0.906990,0.703517,0.973631,0.735260,0.861379,0.785708
4,4,0.843641,0.583819,0.762433,0.899361,0.688610,0.966214,0.729964,0.851395,0.778537
5,5,0.837618,0.590064,0.769408,0.894566,0.657883,0.964607,0.732363,0.839019,0.775026
6,6,0.839701,0.586869,0.771813,0.898885,0.697995,0.967621,0.732794,0.854834,0.781610
7,7,0.835381,0.587245,0.774299,0.892804,0.660371,0.962541,0.732308,0.838572,0.774814
8,8,0.840941,0.586845,0.764231,0.904543,0.706283,0.970383,0.730672,0.860403,0.782564
9,9,0.835618,0.590490,0.783047,0.900455,0.700615,0.971411,0.736385,0.857494,0.784828


In [7]:
score_columns = [
    *SELECTED_METRICS,
    "bio_score",
    "batch_score",
    "joint_score",
]

summary_df = pd.DataFrame(
    {
        "mean": results_df[score_columns].mean(),
        "std": results_df[score_columns].std(ddof=0),
        "min": results_df[score_columns].min(),
        "max": results_df[score_columns].max(),
    }
)
summary_df


,mean,std,min,max
BAMI,0.839492,0.004553,0.831058,0.847728
CASW,0.586681,0.003089,0.580375,0.590490
SCGRAPHM,0.770370,0.008123,0.755485,0.783047
BRAS,0.900891,0.004775,0.892804,0.908906
KBET,0.689131,0.017050,0.657883,0.707044
CPCR,0.969198,0.003592,0.962541,0.973631
bio_score,0.732181,0.004040,0.723320,0.738862
batch_score,0.853073,0.007895,0.838572,0.861379
joint_score,0.780538,0.004511,0.774507,0.787570
